# Phase 1 — Gold & Silver: 100-Year Data Foundation + Baseline Model

**What this notebook does, in order:**
1. Downloads the official LBMA auction prices for **gold and silver from 1968 to today** (free public JSON feed, no key needed).
2. Prepends the **1926–1967** era: gold's official US prices (exact — it was government-fixed) and approximate silver annual averages (labeled as such).
3. Builds and saves a clean **100-year dataset** (`data/` folder) — annual for the full century, monthly from 1968.
4. Trains an honest **baseline forecasting model** on the free-float era (1971→) with walk-forward validation, and compares it against a naive benchmark.

**Why the model does NOT train on all 100 years:** from 1926–1933 gold was fixed at \$20.67/oz and from 1934–1971 at \$35/oz by the US government. A fixed price carries zero predictive signal — training on it would teach the model that gold never moves. The century of data is kept for context and charts; the model learns only from the era when the price was actually set by a market.

**Why this replaces the original fork's approach:** the fork predicted the GLD ETF's *same-day* price from the *same-day* SPX/USO/SLV/EURUSD values. That is not forecasting — the answer is on both sides of the equation, and GLD has only existed since 2004 so it can never reach 100 years. Phase 1 switches to spot prices and to a real question: *given everything known up to month t, what happens in month t+1?*

> Run this in **Google Colab** (fastest: colab.google → GitHub tab → paste this repo's URL) or locally with `pip install pandas scikit-learn matplotlib requests`.


In [ ]:
import json, io, datetime
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
TODAY = datetime.date.today()
print("Run date:", TODAY)


## 1 · LBMA official prices, 1968 → today (gold + silver)

The LBMA publishes its twice-daily auction results as free JSON. `gold_pm.json` is the PM auction (the benchmark most references use); `silver.json` is the single daily silver auction. Each row is `{"d": "YYYY-MM-DD", "v": [USD, GBP, EUR]}` — we take USD.

If the LBMA feed is ever unreachable, the cell falls back to Stooq's spot series (`XAUUSD` / `XAGUSD`) so the notebook still runs — and it prints which source it actually used, so you are never guessing where a number came from.


In [ ]:
def fetch_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "phase1-research"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read().decode())

def fetch_lbma(name):
    raw = fetch_json(f"https://prices.lbma.org.uk/json/{name}.json")
    rows = [(r["d"], r["v"][0]) for r in raw if r.get("v") and r["v"][0] and r["v"][0] > 0]
    df = pd.DataFrame(rows, columns=["date", "usd"])
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index("date").sort_index()

def fetch_stooq(symbol):
    url = f"https://stooq.com/q/d/l/?s={symbol}&i=d"
    req = urllib.request.Request(url, headers={"User-Agent": "phase1-research"})
    with urllib.request.urlopen(req, timeout=60) as r:
        df = pd.read_csv(io.BytesIO(r.read()))
    df["Date"] = pd.to_datetime(df["Date"])
    return df.set_index("Date")[["Close"]].rename(columns={"Close": "usd"}).sort_index()

sources = {}
try:
    gold_d = fetch_lbma("gold_pm");  sources["gold"] = "LBMA PM auction"
except Exception as e:
    print("LBMA gold failed:", e); gold_d = fetch_stooq("xauusd"); sources["gold"] = "Stooq XAUUSD (fallback)"
try:
    silver_d = fetch_lbma("silver"); sources["silver"] = "LBMA auction"
except Exception as e:
    print("LBMA silver failed:", e); silver_d = fetch_stooq("xagusd"); sources["silver"] = "Stooq XAGUSD (fallback)"

print("SOURCES USED:", sources)
print(f"Gold  : {len(gold_d):>6} daily rows, {gold_d.index.min().date()} -> {gold_d.index.max().date()}")
print(f"Silver: {len(silver_d):>6} daily rows, {silver_d.index.min().date()} -> {silver_d.index.max().date()}")


## 2 · The pre-1968 era, 1926–1967

**Gold — exact.** The US official price is a matter of historical record, not estimation: \$20.67/oz through 1932, revalued through 1933 (annual average \$26.33), then fixed at \$35.00 from 1934 until the system broke in 1971.

**Silver — approximate.** Silver traded freely, and the precise year-by-year averages live in the USGS *Historical Statistics for Mineral Commodities* (a free download, but an Excel file that moves URLs). Rather than silently hard-code numbers of uncertain precision, this notebook embeds decade **anchor values** and interpolates between them, and flags every pre-1968 silver row `approx = True`. These rows are used for the century chart only — **never for training** — so their imprecision cannot leak into the model. If you later want them exact, download the USGS silver file and replace the anchors.


In [ ]:
# Gold: official US price, exact
gold_annual_early = {}
for y in range(1926, 1933): gold_annual_early[y] = 20.67
gold_annual_early[1933] = 26.33            # transition-year annual average
for y in range(1934, 1968): gold_annual_early[y] = 35.00

# Silver: decade anchors (approximate annual averages, USD/oz), interpolated
silver_anchors = {1926: 0.62, 1930: 0.38, 1932: 0.28, 1935: 0.64, 1940: 0.35,
                  1945: 0.52, 1950: 0.74, 1955: 0.89, 1960: 0.91, 1965: 1.29, 1967: 1.55}
sa = pd.Series(silver_anchors)
silver_annual_early = sa.reindex(range(1926, 1968)).interpolate(method="linear")

early = pd.DataFrame({
    "gold_usd": pd.Series(gold_annual_early),
    "silver_usd": silver_annual_early,
})
early["approx_silver"] = True
early.index.name = "year"
early.tail(8)


## 3 · Assemble the 100-year dataset and save it

Two files land in `data/`:
- **`gold_silver_100y_annual.csv`** — one row per year, 1926 → today. Annual averages, with an `era` column (`gold_standard`, `bretton_woods`, `free_float`) and the `approx_silver` flag.
- **`gold_silver_monthly_1968.csv`** — month-end averages from the LBMA era, which is what the model actually uses.


In [ ]:
import os
os.makedirs("data", exist_ok=True)

# Monthly averages from the daily auction data
monthly = pd.DataFrame({
    "gold_usd": gold_d["usd"].resample("ME").mean(),
    "silver_usd": silver_d["usd"].resample("ME").mean(),
}).dropna()
monthly.to_csv("data/gold_silver_monthly_1968.csv")

# Annual: modern era from the feeds, early era from the embedded record
modern_annual = pd.DataFrame({
    "gold_usd": gold_d["usd"].resample("YE").mean(),
    "silver_usd": silver_d["usd"].resample("YE").mean(),
}).dropna()
modern_annual.index = modern_annual.index.year
modern_annual["approx_silver"] = False
modern_annual.index.name = "year"

annual = pd.concat([early, modern_annual])
annual = annual[~annual.index.duplicated(keep="last")].sort_index()

def era(y):
    if y < 1934: return "gold_standard"
    if y < 1972: return "bretton_woods"
    return "free_float"
annual["era"] = [era(y) for y in annual.index]
annual["gold_silver_ratio"] = annual["gold_usd"] / annual["silver_usd"]
annual.to_csv("data/gold_silver_100y_annual.csv")

span = annual.index.max() - annual.index.min()
print(f"Annual dataset: {annual.index.min()} -> {annual.index.max()}  ({span} years)")
print(f"Monthly dataset: {monthly.index.min().date()} -> {monthly.index.max().date()}  ({len(monthly)} months)")
annual.tail()


## 4 · The century in one chart

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for a, col, ttl in [(ax[0], "gold_usd", "Gold, USD/oz"), (ax[1], "silver_usd", "Silver, USD/oz")]:
    a.plot(annual.index, annual[col], lw=1.6, color="black")
    a.set_yscale("log"); a.set_title(ttl + "  (log scale)")
    a.axvspan(1926, 1933, alpha=0.12, color="tab:blue",  label="gold standard (fixed)")
    a.axvspan(1934, 1971, alpha=0.12, color="tab:orange", label="Bretton Woods (fixed $35)")
    a.axvspan(1972, int(annual.index.max()), alpha=0.10, color="tab:green", label="free float (model era)")
    a.grid(alpha=0.3)
ax[0].legend(loc="upper left", fontsize=8)
ax[1].set_xlabel("year")
plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 3))
plt.plot(annual.index, annual["gold_silver_ratio"], lw=1.4, color="tab:purple")
plt.title("Gold / silver ratio — how many ounces of silver buy one ounce of gold")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 5 · Baseline model — honest forecasting, free-float era only

**Question posed:** using only information available at the end of month *t*, predict gold's return in month *t+1*.

- **Features (all lagged):** gold and silver returns over the last 1, 3, 6, 12 months; the gold/silver ratio and its 12-month change; 12-month realized volatility; distance from the 12-month moving average.
- **Validation:** walk-forward — train on everything up to year Y, predict year Y, roll forward. The model never sees the future.
- **Benchmarks it must beat:** (a) naive zero (predict no change), (b) always-up. If it cannot beat these, the honest conclusion is that these features carry little monthly signal — a real and common result.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

m = monthly[monthly.index >= "1971-01-01"].copy()
r = m.pct_change()

F = pd.DataFrame(index=m.index)
for lag in (1, 3, 6, 12):
    F[f"gold_ret_{lag}m"]   = m["gold_usd"].pct_change(lag)
    F[f"silver_ret_{lag}m"] = m["silver_usd"].pct_change(lag)
F["gs_ratio"]     = m["gold_usd"] / m["silver_usd"]
F["gs_ratio_chg"] = F["gs_ratio"].pct_change(12)
F["gold_vol_12m"] = r["gold_usd"].rolling(12).std()
F["dist_ma12"]    = m["gold_usd"] / m["gold_usd"].rolling(12).mean() - 1

y = m["gold_usd"].pct_change().shift(-1)          # NEXT month's return
data = F.join(y.rename("target")).dropna()
X, y = data.drop(columns="target"), data["target"]
print(f"{len(data)} monthly samples, {X.shape[1]} features, {data.index.min().date()} -> {data.index.max().date()}")


In [ ]:
preds = []
years = sorted(set(X.index.year))
start = years[0] + 10                              # first decade is training-only
for yr in [v for v in years if v >= start]:
    tr, te = X.index.year < yr, X.index.year == yr
    if te.sum() == 0: continue
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rg = Ridge(alpha=1.0)
    rf.fit(X[tr], y[tr]); rg.fit(X[tr], y[tr])
    p = pd.DataFrame({"actual": y[te],
                      "rf": rf.predict(X[te]),
                      "ridge": rg.predict(X[te])}, index=X.index[te])
    preds.append(p)

P = pd.concat(preds)
P["blend"] = (P["rf"] + P["ridge"]) / 2

def report(name, pred):
    mae = mean_absolute_error(P["actual"], pred)
    hit = (np.sign(pred) == np.sign(P["actual"])).mean()
    return f"{name:<14} MAE {mae:.4f}   directional accuracy {hit:.1%}"

print("Walk-forward, out-of-sample,", P.index.min().date(), "->", P.index.max().date(), f"({len(P)} months)\n")
print(report("RandomForest", P["rf"]))
print(report("Ridge",        P["ridge"]))
print(report("Blend",        P["blend"]))
print(report("Naive (zero)", pd.Series(0.0, index=P.index)))
print(report("Always up",    pd.Series(P['actual'].mean(), index=P.index)))
print(f"\nBase rate: gold rose in {(P['actual']>0).mean():.1%} of months")
P.to_csv("data/phase1_walkforward_predictions.csv")


## 6 · Reading the result honestly

Compare the model rows against the naive rows above. Three outcomes are possible and all three are informative:

- **Model beats naive on MAE and direction** → the lagged features carry real monthly signal. Phase 2 can build on them.
- **Model roughly ties naive** → the most common outcome in academic tests of monthly commodity forecasting. It means *these* features are not enough — the Phase 2 case for adding macro drivers (real interest rates, CPI, the dollar index) writes itself.
- **Model loses to naive** → it is overfitting noise; simplify before adding anything.

What Phase 1 has produced either way: a verified 100-year dataset with its sources and approximations labeled, a leak-free training pipeline, and a measured baseline that any future model must beat to justify its existence.

**Phase 2 candidates:** real-rate and dollar-index features (FRED, free), regime-aware models, monthly-to-weekly step-down, and probability bands instead of point forecasts.
